# Modeling Risk - Quantitative Analysis
**ALY6130 - Group 10**

Monte Carlo Simulation for Quantifying the Pfizer-Seagen Revenue Risk

In [9]:
import random
import pandas as pd
import altair as alt
import numpy as np

In [10]:
import importlib.util

# --- Offline-safe chart rendering ---
# By default Altair renders by loading the Vega JavaScript from a CDN (cdn.jsdelivr.net).
# On an offline or locked-down network that fails with:
#   "Error loading script: https://cdn.jsdelivr.net/npm/vega@5"
# The renderers below avoid the CDN. The most reliable option renders charts to static
# images with the local vl-convert engine. One-time install if needed:
#   pip install vl-convert-python      (then restart the kernel and re-run)
if importlib.util.find_spec("vl_convert") is not None:
    alt.renderers.enable("png")        # static images, fully offline
else:
    alt.renderers.enable("mimetype")   # JupyterLab / VS Code bundled Vega (no CDN)

# Handle large data
alt.data_transformers.enable('default', max_rows=None)

DataTransformerRegistry.enable('default')

# Monte Carlo Simulation for Revenue Risk Profiling

The entire `$43B` Pfizer–Seagen deal thesis rests on a single number: Seagen's **2030 risk-adjusted revenue**, targeted at `$10B`. Rather than commit to one point estimate, we simulate the full range of outcomes for that number by combining three scenarios:

- **Upside** — strong capture of the ADC combination-therapy first-mover opportunity (R-010)
- **Base case** — execution roughly on plan toward the `$10B` target (R-002)
- **Downside** — a revenue miss compounded by IRA price compression (R-003) and biosimilar entry (R-005)

Modeling the whole distribution of outcomes, instead of the single `$10B` headline, gives a much stronger basis for budgeting, contingency planning, and risk communication.

# Case 1

**Note:** Pfizer's investment case assumes Seagen will contribute roughly `$10B` in risk-adjusted revenue by 2030 — `$8B` from its four marketed products plus `$2B` from the pipeline. In 2025 those four products generated only about `$3.46B`, so the climb to `$10B` carries real uncertainty, and a single-point estimate risks over- or under-stating the return. To capture that uncertainty, we use a three-point (PERT) estimate — optimistic, most likely, and pessimistic — to quantify the revenue risk profile. All figures are in millions of dollars. This is a rough first sketch, so feel free to refine the analysis with actual data.

In [11]:
# Three-point estimate for 2030 Seagen revenue ($millions):

optimistic  = random.randint(11000, 12000)   # plan + ADC combination first-mover upside (R-010)
most_likely = random.randint(9000, 10000)    # plan band approaching the $10B target (R-002)
pessimistic = random.randint(6000, 8000)     # revenue miss + IRA price compression (R-003)
print("optimistic =", optimistic, "most_likely =", most_likely, "pessimistic =", pessimistic)
revenue_2030 = (optimistic + 4*most_likely + pessimistic) / 6  # (beta) PERT Three-Point Estimating Technique
print("Single-point PERT estimate ($M):", round(revenue_2030, 1))

optimistic = 11314 most_likely = 9144 pessimistic = 6242
Single-point PERT estimate ($M): 9022.0


In [12]:
# Monte Carlo simulation with 10,000 iterations

np.random.seed(42)   # reproducible results
iteration = 10000

def montecarlo(iteration):
    optimistic  = np.random.randint(11000, 12000, iteration)
    most_likely = np.random.randint(9000, 10000, iteration)
    pessimistic = np.random.randint(6000, 8000, iteration)
    revenue = (optimistic + 4*most_likely + pessimistic) / 6
    return revenue


revenue = montecarlo(iteration)
revenue_df = pd.DataFrame({'Revenue_$M': revenue})
revenue_df

,Revenue_$M
0,9698.000000
1,9248.666667
2,9452.666667
3,9272.833333
4,9546.833333
...,...
9995,9877.833333
9996,9405.000000
9997,9240.333333
9998,9209.833333


In [13]:
# Summary stats to describe the distribution and variability in 2030 revenue

revenue_df.describe()

,Revenue_$M
count,10000.000000
mean,9419.569433
std,220.849240
min,8843.333333
25%,9251.333333
50%,9419.833333
75%,9589.041667
max,9976.000000


In [14]:
# Distribution of simulated 2030 revenue - the tails flag rare upside/downside scenarios to plan for
alt.Chart(revenue_df).mark_bar().encode(
    alt.X('Revenue_$M', bin=alt.Bin(maxbins=50), title='2030 Revenue ($M)'),
    y='count()'
).properties(
    title='2030 Seagen Revenue Distribution_Monte Carlo',
    width=400,
    height=200
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Discussion

The simulated revenue clusters near the PERT mean of about `$9.4B`, with a tight standard deviation of roughly `$0.22B`. Two tails are worth attention:

1. **Rare upside (above ~`$9.7B`)** — strong first-mover capture in ADC combination therapy (R-010) or faster ex-US commercialization. These would help close the gap to the `$10B` thesis but are too unlikely to budget against.
2. **Downside (below ~`$9.1B`)** — a sustained revenue miss (R-002) amplified by IRA price haircuts (R-003). These erode deal ROI and feed the `$17.1B` goodwill-impairment risk (R-012), so they warrant contingency reserves.

Notably, under these base assumptions the simulation essentially never reaches the full `$10B` target, centering instead about 6% below it.

In [15]:
# Cumulative probability

# Here, the cumulative distribution curve helps define the revenue floor to plan around based on
# the proportion of cumulative outcomes. For example, if we want 90% confidence, what revenue can we
# count on being at or above? See below:
revenue_df = revenue_df.sort_values(by='Revenue_$M').reset_index(drop=True)
revenue_df['cumulative_prob'] = (np.arange(len(revenue_df)) + 1) / len(revenue_df)

chart = alt.Chart(revenue_df).mark_line().encode(
    x=alt.X('Revenue_$M', title='2030 Revenue ($M)'),
    y=alt.Y('cumulative_prob', title='Cumulative Probability')
).properties(
    title='2030 Revenue Distribution_Cumulative',
    width=600,
    height=400
).configure_axis(
    titleFontSize=12,
    labelFontSize=10
).configure_title(
    fontSize=14
)

chart

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


# Discussion

The cumulative distribution describes the probabilistic revenue profile of the Seagen acquisition under the PERT assumptions (optimistic, most likely, and pessimistic). Reading it from the downside: roughly **90% of scenarios land at or above ~`$9.1B`** (the 10th percentile), the median is **~`$9.4B`**, and about **90% fall below ~`$9.7B`** (the 90th percentile). Put differently, a risk-adjusted planning figure near **`$9.1B`** carries about 90% confidence, while the headline **`$10B` target falls beyond the simulated range** under these inputs — a quantified view of risk R-002.

# Takeaways

- **Revenue floor:** Use the CDF to anchor a defensible planning figure (e.g., the 10th-percentile ~`$9.1B`) rather than the optimistic headline.
- **Risk appetite:** Match that floor to how much downside the deal economics — debt service and dividend coverage — can absorb.
- **Monte Carlo strength:** It reveals the full spread, not just the average — here, that the `$10B` thesis carries almost no probability mass under base assumptions.
- **Tail planning:** The downside tail (R-002 + R-003 + R-005) is where goodwill impairment and credit-rating pressure originate, so size contingencies and reserves against it.


# Case 2 — Garbage In, Garbage Out (GIGO)

**A Monte Carlo simulation is only as good as the assumptions behind it.** The optimistic, most-likely, and pessimistic inputs therefore have to be quantified with care. In the base case the downside was bounded at `$6–8B`. But several register risks can compound: a revenue miss (R-002), IRA price compression (R-003), and ADCETRIS/PADCEV biosimilar erosion (R-005) striking together could pull revenue toward the 2025 actual of ~`$3.46B`. The cell below re-runs the simulation with that harsher — but still plausible — pessimistic floor to show how much the risk profile shifts. All figures are in millions of dollars.



In [16]:

np.random.seed(42)   # reproducible results
iteration = 10000

def montecarlo(iteration):
    optimistic  = np.random.randint(11000, 12000, iteration)
    most_likely = np.random.randint(9000, 10000, iteration)
    pessimistic = np.random.randint(3000, 7000, iteration)   # compounded downside (R-002 + R-003 + R-005)
    revenue = (optimistic + 4*most_likely + pessimistic) / 6
    return revenue


revenue = montecarlo(iteration)
revenue_df = pd.DataFrame({'Revenue_$M': revenue})

alt.Chart(revenue_df).mark_bar().encode(
    alt.X('Revenue_$M', bin=alt.Bin(maxbins=50), title='2030 Revenue ($M)'),
    y='count()'
).properties(
    title='2030 Revenue Distribution_Harsher Downside (GIGO)',
    width=400,
    height=200
)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


## Discussion

Changing only the pessimistic assumption drops mean revenue from ~`$9.4B` to about **`$9.1B`** and widens the spread. More tellingly, the probability of landing below `$9B` jumps from roughly **2% in the base case to about 39%** here — even though the optimistic and most-likely inputs are unchanged. That is the GIGO lesson: the distribution is only as trustworthy as its inputs, so the three-point estimates — especially the pessimistic tail driven by R-002, R-003, and R-005 — must be carefully justified before the results are relied on.